# Diagonalization Recomposition

Chapter 6: Geometric Transformation and Eigen Decomposition · *Mathematics for AI and Machine Learning* by Fuheng Wu

Rebuild A as PDP^{-1} and inspect reconstruction error.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-6/diagonalization/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.set_printoptions(precision=4)

# The demo's default sliders: eigenvalues and the angle of the second eigenvector
l1, l2, angle_deg = 2.2, 0.9, 50
t = math.radians(angle_deg)

# ── Build A = P D P^{-1} from chosen eigenvectors and eigenvalues ───────────
P = torch.tensor([[1.0, math.cos(t)],
                  [0.0, math.sin(t)]], dtype=torch.float64)   # columns p1, p2
D = torch.diag(torch.tensor([l1, l2], dtype=torch.float64))
A = P @ D @ torch.linalg.inv(P)
print("A = P D P^-1 =\n", A)

# Each column of P is an eigenvector with the chosen eigenvalue
for lam, p in ((l1, P[:, 0]), (l2, P[:, 1])):
    print(f"||A p - {lam} p|| =", (A @ p - lam * p).norm().item())
print("trace(A) =", A.trace().item(), "= l1 + l2 =", l1 + l2)
print("det(A)   =", torch.linalg.det(A).item(), "= l1 * l2 =", l1 * l2)

# ── Go back: eig recovers the eigenvalues (eigenvectors up to scale) ────────
evals, evecs = torch.linalg.eig(A)
print("\neig(A) eigenvalues:", sorted(evals.real.tolist(), reverse=True))

# ── Why it is useful: powers become cheap, A^k = P D^k P^{-1} ───────────────
k = 10
Ak_direct = torch.linalg.matrix_power(A, k)
Ak_diag = P @ torch.diag(torch.tensor([l1 ** k, l2 ** k], dtype=torch.float64)) @ torch.linalg.inv(P)
print(f"A^{k} via P D^{k} P^-1 matches matrix_power:", torch.allclose(Ak_direct, Ak_diag))
print("cond(P) =", torch.linalg.cond(P).item(), "(grows as the eigenvectors become parallel)")

A = P D P^-1 =
 tensor([[ 2.2000, -1.0908],
        [ 0.0000,  0.9000]], dtype=torch.float64)
||A p - 2.2 p|| = 0.0
||A p - 0.9 p|| = 2.220446049250313e-16
trace(A) = 3.1 = l1 + l2 = 3.1
det(A)   = 1.98 = l1 * l2 = 1.9800000000000002

eig(A) eigenvalues: [2.2, 0.8999999999999999]
A^10 via P D^10 P^-1 matches matrix_power: True
cond(P) = 2.1445069205095586 (grows as the eigenvectors become parallel)
